# Experiment on the new teaser format (task YL-004B, lab, 10 Nov)

Three teasers of 9 Nov (Tuta, Pirsch, Cal.com), made with the new Make prompt, and the standard **v3** (`gold_v3.json`, marked by sentence). The same model, the same teasers, **one change between versions**: here the rubric of the judge, `n1` (for the new format) against `n2` (n1 plus a link rating for Company need and Hypothesis lines). Three repeats of each, because the same input can give slightly different answers.

Every run goes into a table and into MLflow (a folder that is downloaded at the end). The MLflow part is not tested yet; if a cell fails, send me the error text.

**Before you start:** the Colab secret `GEMINI_API_KEY` must exist (key icon, "Notebook access" on). The free Gemini tier may be used by Google for training: use it only for public page text.

### Step 1. Code, packages and the MLflow folder
By default the runs are kept in a temporary folder; the last step downloads them. Run the cell again in a new session: it updates the code.

In [ ]:
import os
# MLflow folder (STORE) and the results (OUT: experiments_v2.csv, experiment_runs/, mlflow_overview.html).
# USE_DRIVE = True keeps both on the Drive of this account: Colab then asks for FULL access to the Drive of the
# account that runs this Colab (a Drive of another account cannot be mounted).
USE_DRIVE = True   # this Colab runs in the AURA account: full Drive access is accepted. False = temporary Colab folders
STORE = "/content/mlruns"
OUT = "/content/claim-check/results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        STORE = "/content/drive/MyDrive/teaser_judge_mlruns"
        OUT = "/content/drive/MyDrive/teaser_judge_results"
    except Exception as error:
        print("Google Drive not connected, using the temporary folder:", error)
os.makedirs(STORE, exist_ok=True)
print("MLflow folder:", STORE)
print("Results folder:", OUT)

REPO_URL = "https://github.com/Yul04ek/ai-content-claim-check.git"
BRANCH = "lab-step3"
FOLDER = "/content/claim-check"

# Check for .git, not for the folder: the folder must not exist before the clone (git cannot clone into a non-empty folder)
if not os.path.exists(f"{FOLDER}/.git"):
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
        clone_url = REPO_URL.replace("https://", f"https://{token}@")
    except Exception:
        clone_url = REPO_URL
    !git clone -q -b {BRANCH} {clone_url} {FOLDER}
    !git -C {FOLDER} remote set-url origin {REPO_URL}
else:
    !git -C {FOLDER} fetch -q origin {BRANCH}
    !git -C {FOLDER} checkout -q {BRANCH}
    !git -C {FOLDER} pull -q origin {BRANCH}
%cd {FOLDER}
os.makedirs(OUT, exist_ok=True)   # after the clone: OUT lies inside the repository folder by default
!git log --oneline -1
!pip install -q -r requirements.txt mlflow pandas
import mlflow; print("mlflow", mlflow.__version__)

### Step 2. Connect the model
Should print `ready`. The model must stay `gemini-3.5-flash-lite`, the same as in the first two single runs, or the comparison is not fair.

In [ ]:
import os
from google.colab import userdata

# Read the key from Colab Secrets
try:
    os.environ["API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    raise RuntimeError(
        "Secret GEMINI_API_KEY not found. Add it with the key icon in the left sidebar, "
        "switch on 'Notebook access', and run this cell again."
    )

os.environ["BASE_URL"] = "https://generativelanguage.googleapis.com/v1beta/openai/"  # Gemini through the OpenAI-compatible API
os.environ["MODEL_NAME"] = "gemini-3.5-flash-lite"   # the model that worked in the research agent; change if needed

# Send one short message to check that the key and the model work. Waits and tries again when the model is busy.
import time
from openai import OpenAI, APIStatusError

client = OpenAI(base_url=os.environ["BASE_URL"], api_key=os.environ["API_KEY"]).with_options(timeout=30, max_retries=0)
for attempt in range(5):
    try:
        reply = client.chat.completions.create(
            model=os.environ["MODEL_NAME"],
            messages=[{"role": "user", "content": "Reply with the single word: ready"}],
        )
        print(reply.choices[0].message.content)
        break
    except APIStatusError as error:
        # 503 = the model is overloaded, 429 = per-minute limit. Both are temporary (404 would mean a wrong model name).
        if error.status_code in (429, 500, 502, 503, 504) and attempt < 4:
            wait = 20 * (attempt + 1)
            print(f"Model answered {error.status_code}, waiting {wait} s ...")
            time.sleep(wait)
        else:
            raise

### Step 3. Upload the lab set
One zip file `lab_set_v3_for_colab.zip` (three teasers, their evidence packs, `lab_set.json` and the standard `gold_v3.json`). It stays in Colab only, never in the repository.

In [ ]:
%cd /content/claim-check
import os, glob, json, zipfile
from google.colab import files

uploaded = files.upload()   # choose lab_set_v3_for_colab.zip
os.makedirs("/content/lab_set_v3", exist_ok=True)
for name in uploaded:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall("/content/lab_set_v3")
LAB = glob.glob("/content/lab_set_v3/**/lab_set.json", recursive=True)[0]
print("Lab set:", LAB)
GOLD = os.path.join(os.path.dirname(LAB), json.load(open(LAB)).get("gold", "gold_v2.json"))
print("Standard found:", os.path.exists(GOLD), os.path.basename(GOLD))
assert os.path.exists(GOLD), "the gold file named in lab_set.json is missing: upload lab_set_v3_for_colab.zip"

### Step 4. Free check of the setup (no model call)
Tests, then the code rules alone on the three teasers: `read 0` (no judge) and the number of code findings (it should find 1, 3 and 1 of its problems). Nothing is saved.

In [ ]:
%cd /content/claim-check
!python tests/test_teaser_check.py | tail -1
!python -m teaser_check.experiment "{LAB}" --dry-run --repeats 1

### Step 5. Rubric n1 against n2 (link to the sources), 3 repeats each
n2 is n1 plus one thing: for Company need and Hypothesis lines the judge also says how strongly the sources lead to the sentence (strong / partial / weak / none). The verdict rules are the same, so any difference in the counts comes from that one change. It calls the model 18 times (3 teasers x 3 repeats x 2 rubrics). The model may answer 503: the run waits and tries again, so it can take several minutes.
At the end it prints one table per teaser. In the run folders (`experiment_runs`) lie the result CSV (column `basis`) and the HTML report of every teaser.


In [ ]:
%cd /content/claim-check
!python -m teaser_check.experiment "{LAB}" --rubric skills/experiments/n1.md skills/experiments/n2.md --repeats 3 --store "{STORE}" --out-dir "{OUT}" --experiment teaser-judge-v3


### Step 6. Compare again (optional)
Counts every saved run folder against the standard v3. The same table as at the end of step 5, useful after a restart or after more runs.

In [ ]:
%cd /content/claim-check
!python -m teaser_check.score_v2 "{GOLD}" "{OUT}/experiment_runs" --after 00-00-00

### Step 6b. Company need and Hypothesis sentences across the repeats
One line per sentence: what the gold says, and for every repeat the finding and the link to the sources (`ok/strong`, `no-src/none`, ...). Look at two things: does the finding flip between repeats, and does the link separate the real errors (gold: Unsupported) from the hedged sentences (gold: Needs human). Run it once for n1 (no link given, shows the flipping) and once for n2.


In [ ]:
%cd /content/claim-check
for rubric in ("n1", "n2"):
    print("=== rubric", rubric)
    !python -m teaser_check.basis_report "{GOLD}" "{OUT}/experiment_runs" --rubric {rubric} --out "{OUT}/need_hypothesis_{rubric}.csv"


### Step 6c. NLI check of Company need and Hypothesis (`--pairs`)
A small Hugging Face model, `cross-encoder/nli-deberta-v3-base`, runs on this Colab (first run downloads about 0.7 GB; no text leaves it). It checks only the **Company need** and **Hypothesis** sentences, the ones that are conclusions of the model and not direct quotes. It changes no verdict.
For every offer block of the teaser, each of the two sentences gets three numbers (0 to 1, only entailment is used): against the **Evidence quote alone** (`evidence`), against the **AURA offer alone** (`offer`) and against **both together** (`both`).
- high only `evidence`: the sentence rests on the company's own words;
- high only `offer`: it is probably re-written from the offer;
- high only `both`: the link needs the two sources;
- low everywhere: not supported by either (hedged, Needs human).
Needs a GPU runtime (Runtime, Change runtime type, T4) only to be faster; 14 blocks x 3 passes is also fine on a CPU. Not tested with the real model since the three-number change: if the cell fails, send me the error text. The whole-teaser mode (n1/n2) is paused, see step 6d.

In [ ]:
%cd /content/claim-check
!pip install -q transformers
# Evidence + offer -> Company need and Hypothesis of each block (no judge runs needed)
!python -m teaser_check.nli_score "{GOLD}" --lab "{LAB}" --pairs --out "{OUT}/nli_pairs.csv"

### Step 6d. NLI over all judged statements of the teaser (PAUSED, do not run now)
Decision of 11 Nov: checking the whole teaser this way does not work yet (direct quotes get odd scores, supported statements of How AURA helps score low because no single piece of the sources carries them, errors and hedged sentences are not separated) and has to be rethought. The cell stays here only so that the code is not lost.

In [ ]:
# PAUSED: kept for later, not part of the current work
# %cd /content/claim-check
# for rubric in ("n1", "n2"):
#     print("=== rubric", rubric)
#     !python -m teaser_check.nli_score "{GOLD}" "{OUT}/experiment_runs" --lab "{LAB}" --rubric {rubric} --out "{OUT}/nli_{rubric}.csv"

### Step 7. Save the results
Everything in Colab is deleted when the session ends. This downloads the table, all run folders and the MLflow folder (zip). Send the zip to me.

In [ ]:
%cd /content/claim-check
from google.colab import files
!zip -qr /content/teaser_experiments_v3.zip "{OUT}/experiments_v2.csv" "{OUT}/experiment_runs" "{OUT}/mlflow_overview.csv" "{OUT}/mlflow_overview.html" "{STORE}"
files.download("/content/teaser_experiments_v3.zip")